# Medical Insurance Cost Prediction
### End-to-end AIML Mini Project — Regression

**Objective:** Explore factors associated with personal medical insurance charges and build a model that estimates charges from demographic and lifestyle features.

**Dataset:** Medical Cost Personal Datasets (Kaggle)  
**Kaggle source:** https://www.kaggle.com/datasets/mirichoi0218/insurance  
**Dataset mirror used by this notebook:** [CSV on GitHub](https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv)

> Run the notebook from top to bottom in Google Colab. The notebook downloads a public CSV mirror. If that fails, upload `insurance.csv` when prompted. Model results are calculated when you run the notebook; do not report unexecuted metrics as actual results.

## 1. Problem statement and objectives

Medical insurance charges can vary with factors such as age, body mass index (BMI), smoking status, family size, and region. The goal is to:

- understand the dataset and check its quality;
- explore relationships using at least four meaningful visualizations;
- prepare categorical and numerical features correctly;
- train and compare regression models;
- evaluate predictions using MAE, RMSE, and R²;
- discuss limitations and possible improvements.

**Problem type:** Supervised regression.  
**Target variable:** `charges` (insurance cost in the dataset's currency units).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
print("Libraries imported successfully.")

## 2. Load the dataset

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"

try:
    df = pd.read_csv(DATA_URL)
    print("Loaded dataset from the public CSV mirror.")
except Exception as e:
    print("Automatic download failed:", repr(e))
    try:
        from google.colab import files
        uploaded = files.upload()  # Upload insurance.csv
        filename = next(iter(uploaded))
        df = pd.read_csv(filename)
    except Exception as upload_error:
        raise RuntimeError(
            "Could not download the dataset. Download insurance.csv from "
            "https://www.kaggle.com/datasets/mirichoi0218/insurance and upload it."
        ) from upload_error

print("Dataset shape:", df.shape)
display(df.head())

## 3. Understand the data

In [ ]:
print("Column names:", df.columns.tolist())
print("\nData types and non-null counts:")
df.info()
print("\nNumerical summary:")
display(df.describe(include="all").T)

## 4. Data quality checks

In [ ]:
print("Missing values by column:")
display(df.isna().sum().to_frame("missing_values"))

print("Exact duplicate rows:", int(df.duplicated().sum()))
print("\nUnique values in categorical columns:")
for col in df.select_dtypes(include=["object", "category"]).columns:
    print(f"{col}: {sorted(df[col].dropna().unique().tolist())}")

# Clean column labels and remove exact duplicate records.
df.columns = df.columns.str.strip().str.lower()
before = len(df)
df = df.drop_duplicates().copy()
print(f"Rows before duplicate removal: {before}; after: {len(df)}; removed: {before-len(df)}")

# Confirm expected fields before modelling.
required = {"age", "sex", "bmi", "children", "smoker", "region", "charges"}
missing_columns = required - set(df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing expected columns: {missing_columns}")

# Convert expected numeric columns safely; invalid values become missing and are imputed in pipeline.
for col in ["age", "bmi", "children", "charges"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("\nMissing values after type conversion:")
display(df.isna().sum().to_frame("missing_values"))

### Cleaning decisions

- Checked missing values, duplicate rows, column names, and data types.
- Removed exact duplicate records so they do not receive repeated weight.
- Converted numeric columns using safe numeric conversion.
- Missing feature values, if any, are imputed inside the model pipeline (median for numeric and most frequent for categorical columns).
- The target `charges` is not imputed; rows without a usable target are removed before modelling.

## 5. Exploratory Data Analysis (EDA)

In [ ]:
# Prepare a plotting copy with usable target values.
eda = df.dropna(subset=["charges"]).copy()

# Visualization 1: distribution of insurance charges
plt.figure(figsize=(8, 5))
plt.hist(eda["charges"], bins=30, edgecolor="black")
plt.title("Distribution of Medical Insurance Charges")
plt.xlabel("Charges")
plt.ylabel("Number of people")
plt.tight_layout()
plt.show()
print("Observation: Check whether charges are symmetric or right-skewed; unusually high charges may form a long tail.")

In [ ]:
# Visualization 2: charges by smoking status
groups = [eda.loc[eda["smoker"] == s, "charges"].dropna().values
          for s in sorted(eda["smoker"].dropna().unique())]
labels = sorted(eda["smoker"].dropna().unique())
plt.figure(figsize=(7, 5))
plt.boxplot(groups, tick_labels=labels)
plt.title("Insurance Charges by Smoking Status")
plt.xlabel("Smoker")
plt.ylabel("Charges")
plt.tight_layout()
plt.show()
print("Observation: Compare the medians and spread between smoking groups; this is an association, not proof of causation.")

In [ ]:
# Visualization 3: age vs charges
plt.figure(figsize=(8, 5))
for smoker_value, part in eda.groupby("smoker"):
    plt.scatter(part["age"], part["charges"], alpha=0.55, label=str(smoker_value))
plt.title("Age vs Insurance Charges")
plt.xlabel("Age")
plt.ylabel("Charges")
plt.legend(title="Smoker")
plt.tight_layout()
plt.show()
print("Observation: Look for changes in charges with age and whether the pattern differs by smoking status.")

In [ ]:
# Visualization 4: average charges by region
region_means = eda.groupby("region", observed=False)["charges"].mean().sort_values()
plt.figure(figsize=(8, 5))
plt.bar(region_means.index.astype(str), region_means.values)
plt.title("Average Insurance Charges by Region")
plt.xlabel("Region")
plt.ylabel("Average charges")
plt.tight_layout()
plt.show()
print("Observation: Regional averages may differ, but differences can reflect other features in the dataset.")

In [ ]:
# Visualization 5: BMI vs charges
plt.figure(figsize=(8, 5))
plt.scatter(eda["bmi"], eda["charges"], alpha=0.55)
plt.title("BMI vs Insurance Charges")
plt.xlabel("BMI")
plt.ylabel("Charges")
plt.tight_layout()
plt.show()
print("Observation: Assess whether BMI has a clear linear relationship or whether the spread varies across BMI values.")

## 6. Prepare data and choose models

We predict `charges` using `age`, `sex`, `bmi`, `children`, `smoker`, and `region`.

- A held-out test set is used so evaluation is performed on records not used to fit the models.
- Numeric missing values are filled with the median.
- Categorical missing values are filled with the most frequent value.
- Categorical features are one-hot encoded.
- Two models are compared: Linear Regression (simple baseline) and Random Forest Regressor (can learn non-linear patterns).
- Preprocessing is kept inside a scikit-learn pipeline to reduce data leakage.

In [ ]:
target = "charges"
features = ["age", "sex", "bmi", "children", "smoker", "region"]

model_df = df.dropna(subset=[target]).copy()
X = model_df[features]
y = model_df[target]

numeric_features = ["age", "bmi", "children"]
categorical_features = ["sex", "smoker", "region"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 7. Train and evaluate models

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=300, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1
    )
}

fitted_models = {}
results = []

for name, estimator in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    fitted_models[name] = pipeline
    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "R2": r2_score(y_test, predictions)
    })

results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
display(results_df.style.format({"MAE": "{:,.2f}", "RMSE": "{:,.2f}", "R2": "{:.3f}"}))
best_name = results_df.loc[0, "Model"]
best_model = fitted_models[best_name]
best_predictions = best_model.predict(X_test)
print("Best model by test RMSE:", best_name)
print("Interpretation: lower MAE/RMSE means smaller typical prediction errors; R² closer to 1 indicates more variance explained.")

In [ ]:
# Visualize actual vs predicted values for the selected model.
plt.figure(figsize=(7, 6))
plt.scatter(y_test, best_predictions, alpha=0.65)
low = min(float(y_test.min()), float(np.min(best_predictions)))
high = max(float(y_test.max()), float(np.max(best_predictions)))
plt.plot([low, high], [low, high], linestyle="--")
plt.title(f"Actual vs Predicted Charges — {best_name}")
plt.xlabel("Actual charges")
plt.ylabel("Predicted charges")
plt.tight_layout()
plt.show()

residuals = y_test.to_numpy() - best_predictions
plt.figure(figsize=(8, 5))
plt.scatter(best_predictions, residuals, alpha=0.6)
plt.axhline(0, linestyle="--")
plt.title("Residuals vs Predicted Charges")
plt.xlabel("Predicted charges")
plt.ylabel("Actual - predicted")
plt.tight_layout()
plt.show()

## 8. Feature importance (Random Forest)

In [ ]:
rf_pipeline = fitted_models["Random Forest"]
rf_preprocessor = rf_pipeline.named_steps["preprocessor"]
rf_model = rf_pipeline.named_steps["model"]
feature_names = rf_preprocessor.get_feature_names_out()
importance = pd.Series(rf_model.feature_importances_, index=feature_names).sort_values(ascending=False).head(12)

plt.figure(figsize=(9, 6))
importance.sort_values().plot(kind="barh")
plt.title("Top Feature Importances — Random Forest")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()
display(importance.to_frame("importance"))
print("Feature importance describes how the fitted forest uses features; it does not prove causality or fairness.")

## 9. Conclusion, limitations, and future improvements

In [ ]:
best_row = results_df.iloc[0]
print(f"Selected model by lowest test RMSE: {best_row['Model']}")
print(f"Test MAE: {best_row['MAE']:,.2f}")
print(f"Test RMSE: {best_row['RMSE']:,.2f}")
print(f"Test R²: {best_row['R2']:.3f}")

### Conclusion template — complete after running

After running the notebook, use the printed metrics and charts to write your final conclusion in your own words. State which model performed better on the held-out test set, what patterns the plots showed, and whether the prediction errors seem acceptable for the intended use.

### Limitations

- This is a relatively small, older dataset and may not represent current healthcare costs or all populations.
- Observational relationships do not establish causation.
- Features such as smoking status can be sensitive; predictions should not be used to make real insurance decisions without domain, legal, fairness, and privacy review.
- A single random train/test split gives only one estimate of generalization performance.
- The dataset's charges are not guaranteed to represent a current market quote.

### Future improvements

- Use cross-validation and hyperparameter tuning.
- Compare additional models and inspect errors by subgroup.
- Validate on a larger, newer, representative dataset.
- Review fairness, data governance, and appropriate use before any real-world deployment.

## 10. Project summary

| Requirement | Covered |
|---|---|
| Problem statement and objective | Yes |
| Dataset link and description | Yes |
| Data understanding | Yes |
| Missing values, duplicates, types | Yes |
| At least four visualizations with observations | Yes — five EDA plots plus evaluation plots |
| Regression model and evaluation | Yes — Linear Regression and Random Forest |
| Conclusion, limitations, future work | Yes |

**Before submission:** Run all cells in Colab, check that charts and metrics appear, and personalize the conclusion using your actual outputs. Be ready to explain why you chose regression, what MAE/RMSE/R² mean, and why the results do not prove causation.